In [1]:
import pandas as pd
import numpy as np
import os
import re
import math
import time
import random

In [43]:
df = pd.read_csv("../../Feature_Generation/calculated_features/eins features.csv", index_col=0)
# df.drop("Unnamed: 0", axis=1, inplace=True)
df.head()

,companyCreditHistoryDuration,countryOrRegionOfIncorporationOrRegistration,dateOfChangeOfNameOfCompany,companyPrincipalActivitySubClass,priceOfCompanyIssuedOrdinary,amountOfRevenue,increasedRevenueFromPreviousYear,amountOfProfitAfterTax,isProfitable,increasedProfitFromPreviousYear,...,recentYearCommercialRetailSearchCount,recentYearServicesSearchCount,increaseInCommercialSearchCount,recentYearLegalSearchCount,recentYearLitigationSearchCount,recentYearConveyancingSearchCount,increaseInLegalSearchCount,employeeCount,revenuePerEmployee,profitPerEmployee
entity_ID,,,,,,,,,,,,,,,,,,,,,
01278011T,7862,MALAYSIA,2002-07-03,"Manufacture of blank magnetic tapes, diskettes...",1.023175,10059375481,-63905642558,9223946383,True,-64480052655,...,0,0,-20,0,0,0,-32,15775,6.376783e+05,5.847193e+05
02875427D,7706,VIETNAM,2010-02-26,Armed forces (84221),1.734047,60125171118,-20851601315,59703104371,True,-20901030655,...,0,0,-32,0,0,0,0,20393,2.948324e+06,2.927627e+06
03185739Q,5355,MALAYSIA,2014-05-14,"Shipping companies, including chartering of sh...",0.748938,48315879815,-36873012693,47730672006,True,-37123366354,...,0,0,-16,0,0,0,-8,27785,1.738920e+06,1.717858e+06
04373126M,4610,VIETNAM,2018-05-09,Manufacture of machine tool accessories (e.g. ...,0.699491,85786153589,53566711872,85410671997,True,54092055624,...,0,0,-16,0,0,0,-12,14001,6.127145e+06,6.100327e+06
05255480Z,368,VIETNAM,2022-05-01,Manufacture of wooden containers (16239),1.055539,82813313800,60257496767,82188085219,True,60607679838,...,0,0,-28,0,0,0,-8,34001,2.435614e+06,2.417226e+06


In [44]:
df["target"] = [random.randint(0,1) for i in range(len(df))]
df["target"]

entity_ID
01278011T     0
02875427D     1
03185739Q     1
04373126M     1
05255480Z     1
             ..
T20CX3506S    1
T20LP1292S    1
T21CH6275F    1
T21DP5120D    0
T22RF8858C    1
Name: target, Length: 101, dtype: int64

In [45]:
def is_monotonic(temp_series):
    return all(temp_series[i] <= temp_series[i + 1] for i in range(len(temp_series) - 1)) or all(temp_series[i] >= temp_series[i + 1] for i in range(len(temp_series) - 1))

In [46]:
def prepare_bins(bin_data, c_i, target_col, max_bins):
    force_bin = True
    binned = False
    remarks = np.nan
    # ----------------- Monotonic binning -----------------
    for n_bins in range(max_bins, 2, -1):
        try:
            bin_data[c_i + "_bins"] = pd.qcut(bin_data[c_i], n_bins, duplicates="drop")
            monotonic_series = bin_data.groupby(c_i + "_bins")[target_col].mean().reset_index(drop=True)
            if is_monotonic(monotonic_series):
                force_bin = False
                binned = True
                remarks = "binned monotonically"
                break
        except:
            pass
    # ----------------- Force binning -----------------
    # creating 2 bins forcefully because 2 bins will always be monotonic
    if force_bin or (c_i + "_bins" in bin_data and bin_data[c_i + "_bins"].nunique() < 2):
        _min=bin_data[c_i].min()
        _mean=bin_data[c_i].mean()
        _max=bin_data[c_i].max()
        bin_data[c_i + "_bins"] = pd.cut(bin_data[c_i], [_min, _mean, _max], include_lowest=True)
        if bin_data[c_i + "_bins"].nunique() == 2:
            binned = True
            remarks = "binned forcefully"
    
    if binned:
        return c_i + "_bins", remarks, bin_data[[c_i, c_i+"_bins", target_col]].copy()
    else:
        remarks = "couldn't bin"
        return c_i, remarks, bin_data[[c_i, target_col]].copy()

In [47]:
def iv_woe_4iter(binned_data, target_col, class_col):
    if "_bins" in class_col:
        binned_data[class_col] = binned_data[class_col].cat.add_categories(['Missing'])
        binned_data[class_col] = binned_data[class_col].fillna("Missing")
        temp_groupby = binned_data.groupby(class_col).agg({class_col.replace("_bins", ""):["min", "max"],
                                                           target_col: ["count", "sum", "mean"]}).reset_index()
    else:
        binned_data[class_col] = binned_data[class_col].fillna("Missing")
        temp_groupby = binned_data.groupby(class_col).agg({class_col:["first", "first"],
                                                           target_col: ["count", "sum", "mean"]}).reset_index()
    
    temp_groupby.columns = ["sample_class", "min_value", "max_value", "sample_count", "event_count", "event_rate"]
    temp_groupby["non_event_count"] = temp_groupby["sample_count"] - temp_groupby["event_count"]
    temp_groupby["non_event_rate"] = 1 - temp_groupby["event_rate"]
    temp_groupby = temp_groupby[["sample_class", "min_value", "max_value", "sample_count",
                                 "non_event_count", "non_event_rate", "event_count", "event_rate"]]
    
    if "_bins" not in class_col and "Missing" in temp_groupby["min_value"]:
        temp_groupby["min_value"] = temp_groupby["min_value"].replace({"Missing": np.nan})
        temp_groupby["max_value"] = temp_groupby["max_value"].replace({"Missing": np.nan})
    temp_groupby["feature"] = class_col
    if "_bins" in class_col:
        temp_groupby["sample_class_label"]=temp_groupby["sample_class"].replace({"Missing": np.nan}).astype('category').cat.codes.replace({-1: np.nan})
    else:
        temp_groupby["sample_class_label"]=np.nan
    temp_groupby = temp_groupby[["feature", "sample_class", "sample_class_label", "sample_count", "min_value", "max_value",
                                 "non_event_count", "non_event_rate", "event_count", "event_rate"]]
    
    """
    **********get distribution of good and bad
    """
    temp_groupby['distbn_non_event'] = temp_groupby["non_event_count"]/temp_groupby["non_event_count"].sum()
    temp_groupby['distbn_event'] = temp_groupby["event_count"]/temp_groupby["event_count"].sum()

    temp_groupby['woe'] = np.log(temp_groupby['distbn_non_event'] / temp_groupby['distbn_event'])
    temp_groupby['iv'] = (temp_groupby['distbn_non_event'] - temp_groupby['distbn_event']) * temp_groupby['woe']
    
    temp_groupby["woe"] = temp_groupby["woe"].replace([np.inf,-np.inf],0)
    temp_groupby["iv"] = temp_groupby["iv"].replace([np.inf,-np.inf],0)
    
    return temp_groupby

In [48]:
def var_iter(data, target_col, max_bins):
    woe_iv = pd.DataFrame()
    remarks_list = []
    for c_i in data.columns:
        if c_i not in [target_col]:
            # check if binning is required. if yes, then prepare bins and calculate woe and iv.
            """
            ----logic---
            binning is done only when feature is continuous and non-binary.
            Note: Make sure dtype of continuous columns in dataframe is not object.
            """
            c_i_start_time=time.time()
            if np.issubdtype(data[c_i], np.number) and data[c_i].nunique() > 2:
                class_col, remarks, binned_data = prepare_bins(data[[c_i, target_col]].copy(), c_i, target_col, max_bins)
                agg_data = iv_woe_4iter(binned_data.copy(), target_col, class_col)
                remarks_list.append({"feature": c_i, "remarks": remarks})
            else:
                agg_data = iv_woe_4iter(data[[c_i, target_col]].copy(), target_col, c_i)
                remarks_list.append({"feature": c_i, "remarks": "categorical"})
            # print("---{} seconds. c_i: {}----".format(round(time.time() - c_i_start_time, 2), c_i))
            woe_iv = woe_iv.append(agg_data)
    return woe_iv, pd.DataFrame(remarks_list)

In [49]:
def get_iv_woe(data, target_col, max_bins):
    func_start_time = time.time()
    woe_iv, binning_remarks = var_iter(data, target_col, max_bins)
    print("------------------IV and WOE calculated for individual groups.------------------")
    print("Total time elapsed: {} minutes".format(round((time.time() - func_start_time) / 60, 3)))
    
    woe_iv["feature"] = woe_iv["feature"].replace("_bins", "", regex=True)    
    woe_iv = woe_iv[["feature", "sample_class", "sample_class_label", "sample_count", "min_value", "max_value",
                     "non_event_count", "non_event_rate", "event_count", "event_rate", 'distbn_non_event',
                     'distbn_event', 'woe', 'iv']]
    
    iv = woe_iv.groupby("feature")[["iv"]].agg(["sum", "count"]).reset_index()
    print("------------------Aggregated IV values for features calculated.------------------")
    print("Total time elapsed: {} minutes".format(round((time.time() - func_start_time) / 60, 3)))
    
    iv.columns = ["feature", "iv", "number_of_classes"]
    null_percent_data=pd.DataFrame(data.isnull().mean()).reset_index()
    null_percent_data.columns=["feature", "feature_null_percent"]
    iv=iv.merge(null_percent_data, on="feature", how="left")
    print("------------------Null percent calculated in features.------------------")
    print("Total time elapsed: {} minutes".format(round((time.time() - func_start_time) / 60, 3)))
    iv = iv.merge(binning_remarks, on="feature", how="left")
    woe_iv = woe_iv.merge(iv[["feature", "iv", "remarks"]].rename(columns={"iv": "iv_sum"}), on="feature", how="left")
    print("------------------Binning remarks added and process is complete.------------------")
    print("Total time elapsed: {} minutes".format(round((time.time() - func_start_time) / 60, 3)))
    return iv, woe_iv.replace({"Missing": np.nan})

In [50]:
iv, woe_iv = get_iv_woe(df.copy(), target_col="target", max_bins=20)

C:\Users\brian\AppData\Local\Temp\ipykernel_9720\815430266.py:21: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  woe_iv = woe_iv.append(agg_data)
C:\Users\brian\AppData\Local\Temp\ipykernel_9720\2063493339.py:9: PerformanceWarning: indexing past lexsort depth may impact performance.
  temp_groupby = binned_data.groupby(class_col).agg({class_col:["first", "first"],
C:\Users\brian\AppData\Local\Temp\ipykernel_9720\815430266.py:21: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  woe_iv = woe_iv.append(agg_data)
C:\Users\brian\AppData\Local\Temp\ipykernel_9720\2063493339.py:9: PerformanceWarning: indexing past lexsort depth may impact performance.
  temp_groupby = binned_data.groupby(class_col).agg({class_col:["first", "first"],
c:\Users\brian\AppData\Local\Programs\Python\Python38\lib\site-packages\pandas\core\arrayl

------------------IV and WOE calculated for individual groups.------------------
Total time elapsed: 0.034 minutes
------------------Aggregated IV values for features calculated.------------------
Total time elapsed: 0.034 minutes
------------------Null percent calculated in features.------------------
Total time elapsed: 0.034 minutes
------------------Binning remarks added and process is complete.------------------
Total time elapsed: 0.034 minutes


C:\Users\brian\AppData\Local\Temp\ipykernel_9720\815430266.py:21: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  woe_iv = woe_iv.append(agg_data)


In [51]:
iv

,feature,iv,number_of_classes,feature_null_percent,remarks
0,amountOfProfitAfterTax,0.009939,2,0.0,binned forcefully
1,amountOfRevenue,0.061898,3,0.0,binned monotonically
2,amountOfShareholdersFund,0.061898,3,0.0,binned monotonically
3,auditor,0.107997,4,0.0,categorical
4,companyCreditHistoryDuration,0.048495,5,0.0,binned monotonically
...,...,...,...,...,...
80,totalWOSAmountAgainstRevenue,0.000000,1,0.0,categorical
81,totalWOSAndOCAgainstRevenue,0.000000,1,0.0,categorical
82,totalWOSAndOCAmount,0.000000,1,0.0,categorical
83,totalWindingUpAmount,0.038260,3,0.0,binned monotonically


In [52]:
woe_iv

,feature,sample_class,sample_class_label,sample_count,min_value,max_value,non_event_count,non_event_rate,event_count,event_rate,distbn_non_event,distbn_event,woe,iv,iv_sum,remarks
0,companyCreditHistoryDuration,"(161.999, 1808.0]",0.0,21,162.0,1808.0,12,0.571429,9,0.428571,0.24,0.176471,0.307485,0.019534,0.048495,binned monotonically
1,companyCreditHistoryDuration,"(1808.0, 3203.0]",1.0,20,1825.0,3203.0,11,0.550000,9,0.450000,0.22,0.176471,0.220473,0.009597,0.048495,binned monotonically
2,companyCreditHistoryDuration,"(3203.0, 5418.0]",2.0,20,3214.0,5418.0,9,0.450000,11,0.550000,0.18,0.215686,-0.180868,0.006455,0.048495,binned monotonically
3,companyCreditHistoryDuration,"(5418.0, 7343.0]",3.0,20,5462.0,7343.0,9,0.450000,11,0.550000,0.18,0.215686,-0.180868,0.006455,0.048495,binned monotonically
4,companyCreditHistoryDuration,"(7343.0, 8492.0]",4.0,20,7433.0,8492.0,9,0.450000,11,0.550000,0.18,0.215686,-0.180868,0.006455,0.048495,binned monotonically
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
839,revenuePerEmployee,"(2020951.517, 7067577.004]",1.0,32,2109040.222379,7067577.004035,16,0.500000,16,0.500000,0.32,0.313725,0.019803,0.000124,0.000182,binned forcefully
840,revenuePerEmployee,NaN,NaN,0,NaN,NaN,0,NaN,0,NaN,0.00,0.000000,NaN,NaN,0.000182,binned forcefully
841,profitPerEmployee,"(205313.11800000002, 1996898.494]",0.0,69,205313.119255,1990460.429812,34,0.492754,35,0.507246,0.68,0.686275,-0.009185,0.000058,0.000182,binned forcefully
842,profitPerEmployee,"(1996898.494, 7028979.239]",1.0,32,2077774.37077,7028979.239072,16,0.500000,16,0.500000,0.32,0.313725,0.019803,0.000124,0.000182,binned forcefully


In [53]:
selected_iv = iv[iv["iv"]>0.02]
selected_iv = selected_iv[selected_iv["remarks"]!="categorical"]
selected_iv

,feature,iv,number_of_classes,feature_null_percent,remarks
1,amountOfRevenue,0.061898,3,0.0,binned monotonically
2,amountOfShareholdersFund,0.061898,3,0.0,binned monotonically
4,companyCreditHistoryDuration,0.048495,5,0.0,binned monotonically
5,companyDebtRatio,0.071561,3,0.0,binned monotonically
7,companyNetProfitMarginAgainstIndustryMedian,0.114141,2,0.0,binned forcefully
11,companyReturnOnAssetsAgainstIndustryMedian,0.071561,3,0.0,binned monotonically
18,employeeCount,0.085571,3,0.0,binned monotonically
25,increaseInCommercialSearchCount,0.067489,2,0.0,binned forcefully
26,increaseInFinancialSearchCount,0.053182,2,0.0,binned forcefully
28,increasedDebtRatioFromPreviousYear,0.032094,2,0.0,binned forcefully


In [54]:
from varclushi import VarClusHi

clus_df = df[selected_iv["feature"].values.tolist()]
var_clus_model = VarClusHi(clus_df, maxeigval2 = 0.7, maxclus=20)
var_clus_model.varclus()

In [55]:
var_clus_df = var_clus_model.rsquare
var_clus_df["iv"] = [selected_iv[selected_iv["feature"]==i]["iv"].values[0] for i in var_clus_df["Variable"]]
#var_clus_df["corr"] = [np.corrcoef(df["target"], df[c])[0,1] for c in var_clus_df["Variable"]]
var_clus_df.head()

,Cluster,Variable,RS_Own,RS_NC,RS_Ratio,iv
0,0,amountOfRevenue,0.823194,0.309719,0.256136,0.061898
1,0,amountOfShareholdersFund,0.823296,0.309747,0.255999,0.061898
2,0,increasedProfitFromPreviousYear,0.930531,0.216250,0.088637,0.049008
3,0,increasedRevenueFromPreviousYear,0.930476,0.215176,0.088586,0.049008
4,0,increasedShareholdersFundFromPreviousYear,0.930426,0.215117,0.088642,0.049008


In [56]:
var_clus_df.sort_values(by=["Cluster", "iv"], ascending=[True, False], inplace=True)
var_clus_df.head()

,Cluster,Variable,RS_Own,RS_NC,RS_Ratio,iv
0,0,amountOfRevenue,0.823194,0.309719,0.256136,0.061898
1,0,amountOfShareholdersFund,0.823296,0.309747,0.255999,0.061898
2,0,increasedProfitFromPreviousYear,0.930531,0.216250,0.088637,0.049008
3,0,increasedRevenueFromPreviousYear,0.930476,0.215176,0.088586,0.049008
4,0,increasedShareholdersFundFromPreviousYear,0.930426,0.215117,0.088642,0.049008


In [57]:
var_clus_df.groupby('Cluster').head(1)

,Cluster,Variable,RS_Own,RS_NC,RS_Ratio,iv
0,0,amountOfRevenue,0.823194,0.309719,2.561357e-01,0.061898
5,1,totalOSAmountAgainstRevenue,1.000000,0.355590,0.000000e+00,0.105096
7,2,numShareholders,0.996447,0.040313,3.701965e-03,0.050603
9,3,companyDebtRatio,0.877860,0.032209,1.262052e-01,0.071561
11,4,companyNetProfitMarginAgainstIndustryMedian,0.770695,0.034910,2.375995e-01,0.114141
13,5,increaseInFinancialSearchCount,1.000000,0.035644,0.000000e+00,0.053182
14,6,numExistingLoans,1.000000,0.028297,0.000000e+00,0.062610
15,7,employeeCount,1.000000,0.026952,0.000000e+00,0.085571
16,8,totalWindingUpAmount,1.000000,0.268091,0.000000e+00,0.038260
17,9,increaseInCommercialSearchCount,1.000000,0.038133,0.000000e+00,0.067489


In [58]:
selected_var_list = var_clus_df.groupby('Cluster').head(1)["Variable"].values.tolist()
selected_var_list

['amountOfRevenue',
 'totalOSAmountAgainstRevenue',
 'numShareholders',
 'companyDebtRatio',
 'companyNetProfitMarginAgainstIndustryMedian',
 'increaseInFinancialSearchCount',
 'numExistingLoans',
 'employeeCount',
 'totalWindingUpAmount',
 'increaseInCommercialSearchCount',
 'companyReturnOnAssetsAgainstIndustryMedian',
 'companyCreditHistoryDuration',
 'totalOSAmount',
 'windingUpTotalAmountAgainstRevenue']

In [59]:
selected_var_df = var_clus_df.groupby('Cluster').head(1)
selected_var_df.to_csv('../selected_features/eins_selected_features.csv', index=False)